# Stage 3 Prompt v2 AB Run (CPU fallback)
Full val run with prompt v2 on CPU fallback for compatibility.

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

def sh(cmd: str, cwd: str | None = None):
    print(f'$ {cmd}')
    p = subprocess.run(cmd, shell=True, cwd=cwd, text=True, capture_output=True)
    if p.stdout:
        print(p.stdout)
    if p.stderr:
        print(p.stderr)
    if p.returncode != 0:
        raise RuntimeError(f'Command failed: {cmd}')
    return p

REPO_DIR = Path('/kaggle/working/vlm-for-insulator-defect-detection')
if not REPO_DIR.exists():
    sh('git clone --depth 1 https://github.com/konstRyaz/vlm-for-insulator-defect-detection.git /kaggle/working/vlm-for-insulator-defect-detection')
else:
    print('Repo already exists')

sh('python -m pip install -q -U transformers accelerate qwen-vl-utils')
os.chdir(REPO_DIR)
print('cwd:', os.getcwd())


In [ ]:
VAL_JSONL = Path('/kaggle/input/idid-coco-v3/stage3_regrouped_v2/val/vlm_labels_v1_val_v2.annotated.jsonl')
if not VAL_JSONL.exists():
    found = list(Path('/kaggle/input').rglob('vlm_labels_v1_val_v2.annotated.jsonl'))
    if not found:
        raise FileNotFoundError('Could not resolve val jsonl in /kaggle/input')
    VAL_JSONL = found[0]
print('VAL_JSONL:', VAL_JSONL)


In [ ]:
import yaml
base_cfg = REPO_DIR / 'configs/pipeline/stage3_vlm_gt_baseline.yaml'
runtime_cfg = REPO_DIR / 'configs/pipeline/stage3_vlm_gt_baseline.kaggle_cpu.yaml'
cfg = yaml.safe_load(base_cfg.read_text(encoding='utf-8'))
cfg.setdefault('backend', {}).setdefault('qwen_hf', {})
cfg['backend']['qwen_hf']['torch_dtype'] = 'float32'
cfg['backend']['qwen_hf']['attn_implementation'] = None
cfg['backend']['qwen_hf']['device_map'] = 'cpu'
runtime_cfg.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding='utf-8')
print('runtime cfg:', runtime_cfg)


In [ ]:
run_id = 'stage3_qwen_val_v2_prompt_v2'
os.environ['CUDA_VISIBLE_DEVICES'] = ''
sh(
    'python scripts/run_stage3_vlm_baseline.py ' +
    f'--config \"{runtime_cfg}\" ' +
    '--backend-mode qwen_hf ' +
    '--prompt-version qwen_vlm_labels_v1_prompt_v2 ' +
    f'--input-jsonl \"{VAL_JSONL}\" ' +
    f'--run-id {run_id} --no-resume',
    cwd=str(REPO_DIR),
)


In [ ]:
run_dir = REPO_DIR / 'outputs' / 'stage3_vlm_baseline_runs' / 'stage3_qwen_val_v2_prompt_v2'
pred_jsonl = run_dir / 'predictions_vlm_labels_v1.jsonl'
sh(f'python scripts/validate_vlm_labels_v1.py --input "{pred_jsonl}"', cwd=str(REPO_DIR))
sh(
    f'python scripts/eval_stage3_vlm_baseline.py --run-dir "{run_dir}" --ground-truth-jsonl "{VAL_JSONL}"',
    cwd=str(REPO_DIR),
)
print('run_dir:', run_dir)


In [ ]:
art_dir = Path('/kaggle/working/stage3_prompt_v2_artifacts')
if art_dir.exists():
    shutil.rmtree(art_dir)
art_dir.mkdir(parents=True, exist_ok=True)

to_copy = [
    run_dir / 'run_summary.json',
    run_dir / 'config_snapshot.json',
    run_dir / 'raw_responses.jsonl',
    run_dir / 'parsed_predictions.jsonl',
    run_dir / 'predictions_vlm_labels_v1.jsonl',
    run_dir / 'failures.jsonl',
    run_dir / 'sample_results.jsonl',
    run_dir / 'eval/metrics.json',
    run_dir / 'eval/confusion_coarse_class.csv',
    run_dir / 'eval/confusion_visibility.csv',
    run_dir / 'eval/review_table.csv',
    run_dir / 'eval/failures.jsonl',
]

for p in to_copy:
    if p.exists():
        dest = art_dir / p.relative_to(run_dir)
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(p, dest)

print('artifact dir:', art_dir)
for x in sorted(art_dir.rglob('*')):
    if x.is_file():
        print(x.relative_to(art_dir))


In [ ]:
# Reduce kernel output size
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR, ignore_errors=True)
print('Cleanup done')
